# Actividad 3 - "Construyendo el Cerebro: Keras Sequential API"

**Objetivo:** Ya sabemos mover datos con `tf.data`. Hoy vamos a construir nuestra primera Red Neuronal Profunda utilizando la API Secuencial de Keras. 

Vamos a trabajar con un dataset real de Spotify. Analizaremos las 10 características acústicas de miles de canciones para predecir si la canción será un éxito comercial y a qué género musical pertenece.

### Preparación del Entorno
Ejecuta esta celda para descargar el dataset original de Spotify y filtrar las características acústicas numéricas que utilizará nuestra red neuronal. **(No modifiques esta celda)**.

In [19]:
import tensorflow as tf
import pandas as pd
import numpy as np

# 1. Descargamos el dataset
url_spotify = 'https://raw.githubusercontent.com/sushmaakoju/spotify-tracks-data-analysis/main/SpotifyFeatures.csv'
print("Descargando datos de Spotify...")
df_spotify = pd.read_csv(url_spotify)

# 2. Creamos el target Binario (Hit: 1 si popularidad > 50, sino 0)
df_spotify['target'] = (df_spotify['popularity'] > 50).astype(int)

# 3. Filtramos los 4 géneros más comunes para el problema Multiclase
top_4_generos = df_spotify['genre'].value_counts().nlargest(4).index.tolist()
df_spotify = df_spotify[df_spotify['genre'].isin(top_4_generos)].copy()

# Extraemos las categorías para mapear el ID al nombre del género después
categoria_genero = df_spotify['genre'].astype('category')
nombres_generos = categoria_genero.cat.categories.tolist()
df_spotify['genero_id'] = categoria_genero.cat.codes

# 4. Seleccionamos las 10 características matemáticas
features_acusticas = ['acousticness', 'danceability', 'duration_ms', 'energy', 
                      'instrumentalness', 'liveness', 'loudness', 'speechiness', 
                      'tempo', 'valence']

df_final = df_spotify[features_acusticas + ['target', 'genero_id']].dropna()

print(f"\nDataset listo: {len(df_final)} canciones procesadas.")
print(f"Géneros a clasificar: {nombres_generos} (IDs: 0, 1, 2, 3)")
df_final.head(3)

Descargando datos de Spotify...

Dataset listo: 38311 canciones procesadas.
Géneros a clasificar: ['Comedy', 'Indie', 'Jazz', 'Soundtrack'] (IDs: 0, 1, 2, 3)


,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,target,genero_id
92824,0.29700,0.752,201661,0.488,0.000009,0.0936,-7.050,0.0705,136.041,0.533,1,1
92825,0.01160,0.464,239507,0.852,0.000000,0.1080,-3.804,0.0670,160.075,0.233,1,1
92826,0.00847,0.560,218013,0.936,0.000000,0.1610,-5.835,0.0439,112.960,0.371,1,1


--- 
### Parte 1: El Predictor de "Hits"
Crearemos una red neuronal que reciba las 10 características musicales y prediga si será un éxito comercial (1) o no (0). Estamos ante un problema de **Clasificación Binaria**.

**Pasos:**
1. Añade una capa oculta `Dense` de 16 neuronas, activación `relu` y un `input_shape` adecuado para nuestras 10 columnas.
2. Añade la capa de salida. Piensa cuántas neuronas necesitas para un Sí/No y qué función de activación usar.

In [20]:
modelo_hit = tf.keras.Sequential([
    # Rellena los huecos (___)
    tf.keras.layers.Dense(16, activation='relu', input_shape=(10,)),
    
    # Capa de Salida para Clasificación Binaria
    tf.keras.layers.Dense(1, activation='sigmoid')
])

modelo_hit.summary()

c:\Users\jperaltaza\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_28 (Dense)                │ (None, 16)             │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193 (772.00 B)

 Trainable params: 193 (772.00 B)

 Non-trainable params: 0 (0.00 B)

--- 
### Parte 2: El Clasificador de Géneros
El objetivo ahora es predecir el **género exacto** de la canción (4 géneros posibles). Se trata de un problema de **Clasificación Multiclase**.

**Arquitectura requerida:**
- Capa oculta 1: 32 neuronas, activación relu, entrada de 10 variables.
- Capa oculta 2: 16 neuronas, activación relu.
- Capa de salida: Configurada para 4 clases y activación que genere distribuciones de probabilidad.

In [21]:
modelo_generos = tf.keras.Sequential()

# Añade aquí la primera capa oculta (¡No olvides el input_shape!)
modelo_generos.add(tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)))

# Añade aquí la segunda capa oculta
modelo_generos.add(tf.keras.layers.Dense(16, activation='relu'))

# Añade aquí la capa de salida para Clasificación Multiclase
modelo_generos.add(tf.keras.layers.Dense(4, activation='softmax'))

modelo_generos.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_30 (Dense)                │ (None, 32)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 948 (3.70 KB)

 Trainable params: 948 (3.70 KB)

 Non-trainable params: 0 (0.00 B)

--- 
### Parte 3: El Desastre - Entrenando con datos crudos
Tenemos el modelo y los datos. Vamos a entrenarlo durante **10 épocas** para observar el comportamiento de la red cuando se alimenta con datos en diferentes escalas matemáticas.

Sigue las instrucciones en los comentarios para construir el pipeline, compilar y entrenar.

In [23]:
# 1. Separación de características (X) y etiquetas (y)
X = df_final.drop(['target', 'genero_id'], axis=1).values
y = df_final['genero_id'].values

# 2. Crea el pipeline de tf.data usando X e y (batch de 32 y prefetch AUTOTUNE)
dataset_roto = tf.data.Dataset.from_tensor_slices((X, y)).batch(32).prefetch(tf.data.AUTOTUNE)

# 3. Compila el 'modelo_generos' (optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo_generos.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Entrena el modelo usando .fit() durante 10 épocas y guárdalo en la variable 'historial_roto'
print("\n--- ENTRENANDO MODELO CON DATOS CRUDOS (10 épocas) ---")
historial_roto = modelo_generos.fit(dataset_roto, epochs=10)



--- ENTRENANDO MODELO CON DATOS CRUDOS (10 épocas) ---
Epoch 1/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9466 - loss: 994.1769
Epoch 2/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 991us/step - accuracy: 0.7055 - loss: 464.1925 
Epoch 3/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.2463 - loss: 1.4280
Epoch 4/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2470 - loss: 1.4036
Epoch 5/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.2537 - loss: 1.3973
Epoch 6/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2571 - loss: 1.3951
Epoch 7/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2596 - loss: 1.3942
Epoch 8/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2604 - loss: 1.3938
Epoch 9/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2604 - loss: 1.3936
Epoch 10/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 985us/step - accuracy: 0.2612 - loss: 1.3935


--- 
### Parte 4: El Arreglo Mágico - Normalizando los datos
¿Se ha atascado el modelo en ~25% de precisión? Ocurre porque variables gigantes (como `duration_ms`) desestabilizan las matemáticas de la red.

**Misión:** Usar `StandardScaler` de la librería `scikit-learn` para estandarizar todas las características numéricas a una escala común. Después, entrenaremos un modelo idéntico desde cero para ver la magia.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Instancia el StandardScaler y aplica fit_transform a tus datos X originales
scaler = 
X_normalizado = 

# 2. Crea un NUEVO pipeline llamado 'dataset_arreglado' con X_normalizado e y
dataset_arreglado = 

# 3. Vuelve a crear el modelo desde cero para reiniciar sus pesos
modelo_arreglado = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax')
])

# 4. Compila el 'modelo_arreglado' igual que antes


# 5. Entrena el modelo corregido con dataset_arreglado durante 20 épocas
print("\n--- ENTRENANDO MODELO CON DATOS NORMALIZADOS (20 épocas) ---")
historial_arreglado = 


--- 
### Parte 5: ¿Más grande es mejor?
Construye un modelo llamado `modelo_profundo` con 3 capas ocultas amplias: **128, 64 y 32 neuronas**. (Recuerda la capa de salida).
Compílalo y entrénalo durante **50 épocas** usando el dataset normalizado.

Observa la métrica `accuracy`. ¿Es la red genuinamente más inteligente o simplemente está memorizando el dataset de entrenamiento?

In [ ]:
# TU CÓDIGO AQUÍ:
# - Construye modelo_profundo
# - Compílalo
# - Entrénalo 50 épocas con los datos arreglados




--- 
### 🎧 Parte 6: Predicción de un nuevo Género
Supongamos que acabas de producir una nueva canción. Tienes sus características extraídas. 

**Atención:** Para que la red la entienda, tienes que aplicar la misma transformación (escalado) a esta nueva canción antes de dársela a `predict()`. Usa el `scaler` que ya has entrenado antes.

In [ ]:
# Características de la nueva pista (orden de columnas mantenido)
cancion_estudio = np.array([[
    0.1,    # acousticness
    0.9,    # danceability
    210000, # duration_ms
    0.8,    # energy
    0.0,    # instrumentalness
    0.2,    # liveness
    -5.0,   # loudness
    0.05,   # speechiness
    130.0,  # tempo
    0.8     # valence
]])

# TU CÓDIGO AQUÍ:
# 1. Transforma (escala) 'cancion_estudio' usando scaler.transform()


# 2. Usa modelo_arreglado para predecir las probabilidades de la canción normalizada


# 3. Usa np.argmax para sacar el ID del género ganador


# Descomenta estas líneas cuando tengas tu variable genero_ganador_id lista:
# nombre_genero = nombres_generos[genero_ganador_id]
# print(f"\nEl modelo predice que la pista pertenece al género: {nombre_genero} (ID: {genero_ganador_id})")


--- 
### Parte 7: RETO FINAL - El Proyecto Completo

En la **Parte 1** construimos el diseño de una red para predecir si una canción sería un **Éxito Comercial (Hit)** o no, pero... nunca llegamos a entrenarla.

**Tu misión:** Demuestra todo lo que has aprendido. Construye el flujo de trabajo completo para este problema de **Clasificación Binaria**.

**Especificaciones:**
1. **Datos:** Las variables de entrada (`X`) son las mismas, pero la variable a predecir ahora es `df_final['target']`.
2. **Preprocesamiento:** Aplica la estandarización necesaria (¡crea un scaler nuevo para no pisar el anterior!).
3. **Pipeline:** Construye un nuevo `tf.data.Dataset`.
4. **Modelo:** Diseña la red. Usa las capas ocultas que quieras, pero recuerda que **la capa de salida debe ser de clasificación binaria**.
5. **Compilación:** ¡Atención! Al ser un problema binario, la función de pérdida a utilizar es `binary_crossentropy`.
6. **Entrenamiento:** Entrena el modelo durante 20 épocas.
7. **Inferencia:** Pasa la `cancion_estudio` por este nuevo modelo (escalándola primero) y determina si será un Hit (> 0.5 de probabilidad).

In [ ]:
# Ánimo :)

